In [8]:
# train_model_augmented.py
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image, UnidentifiedImageError
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
import joblib

# Paths
DATA_DIR = r"C:\Users\priya\Downloads\Smart_Nutrition\nutrients\images"
NUTRITION_CSV = r"C:\Users\priya\Downloads\Smart_Nutrition\nutrients\data.csv"
MODEL_SAVE_PATH = r"C:\Users\priya\Downloads\Smart_Nutrition\models\food_model.h5"
SCALERS_PATH = r"C:\Users\priya\Downloads\Smart_Nutrition\models\nutrient_scalers.pkl"

# Load nutrition CSV
nutrition_df = pd.read_csv(NUTRITION_CSV)
nutrition_df['image'] = nutrition_df['image'].str.strip()

# Nutrients to predict
nutrient_columns = ['carbs','fat','fibre','kcal','protein','salt','saturates','sugars']

# ================= SCALE NUTRIENTS =================
scalers = {}
for col in nutrient_columns:
    scaler = MinMaxScaler(feature_range=(0, 100))
    nutrition_df[col + "_scaled"] = scaler.fit_transform(nutrition_df[[col]])
    scalers[col] = scaler

# Save scalers for later use
os.makedirs(os.path.dirname(SCALERS_PATH), exist_ok=True)
joblib.dump(scalers, SCALERS_PATH)
print("✅ Scalers saved!")

# Load images safely
def load_data(df, folder=DATA_DIR):
    images = []
    nutrients = []
    skipped = 0
    for idx, row in df.iterrows():
        img_path = os.path.join(folder, row['image'])
        if os.path.exists(img_path):
            try:
                with Image.open(img_path) as img:
                    img = img.convert('RGB')          # Ensure 3 channels
                    img = img.resize((224,224))       # Resize to model input
                    img_array = np.array(img)/255.0
                    images.append(img_array)
                    # Use scaled nutrient values
                    nutrients.append(row[[col + "_scaled" for col in nutrient_columns]].values.astype(float))
            except (UnidentifiedImageError, OSError) as e:
                print(f"Skipped invalid image: {row['image']} -> {e}")
                skipped += 1
        else:
            print(f"Image not found: {row['image']}")
            skipped += 1
    print(f"Total skipped images: {skipped}")
    return np.array(images), np.array(nutrients, dtype=float)

# Load data
X, y = load_data(nutrition_df)
print("Images loaded:", X.shape)
print("Nutrients loaded:", y.shape)

# Split data
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ================= DATA AUGMENTATION =================
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator()  # Only rescale if needed

# Build model with EfficientNetB0
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(len(nutrient_columns), activation='linear', name='nutrients')(x)

model = models.Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Callbacks
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True)
]

# Train model using augmentation
history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=16),
    validation_data=val_datagen.flow(X_val, y_val, batch_size=16),
    epochs=50,
    callbacks=callbacks
)

model.save(MODEL_SAVE_PATH)
print("✅ Model saved at:", MODEL_SAVE_PATH)

✅ Scalers saved!
Skipped invalid image: roasted-summer-veg-casserole-3c459e9.png -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\roasted-summer-veg-casserole-3c459e9.png'
Skipped invalid image: Jerk-Style-Cauliflower-With-Coconut-Rice-2c54f98.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Jerk-Style-Cauliflower-With-Coconut-Rice-2c54f98.jpg'
Skipped invalid image: Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg'


C:\Users\priya\anaconda3\Lib\site-packages\PIL\Image.py:996: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Skipped invalid image: Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg'


C:\Users\priya\anaconda3\Lib\site-packages\PIL\Image.py:996: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Skipped invalid image: KimchiPancakes-e591d25.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\KimchiPancakes-e591d25.jpg'
Skipped invalid image: TteokbokkiSpicyRiceCakes-b78e346.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\TteokbokkiSpicyRiceCakes-b78e346.jpg'
Skipped invalid image: Vegan-carbonara-ebf05ee.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Vegan-carbonara-ebf05ee.jpg'
Skipped invalid image: Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Giant-Couscous-Salad-With-Charred-Veg-Tangy-Pesto-aea3737.jpg'
Skipped invalid image: Vegan-carbonara-ebf05ee.jpg -> cannot identify image file 'C:\\Users\\priya\\Downloads\\Smart_Nutrition\\nutrients\\images\\Vegan-carbonara-ebf05ee.jpg'
Total skipped images: 9
Images loade

C:\Users\priya\anaconda3\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


63/63 [==============================] - 88s 1s/step - loss: 267.8987 - mae: 12.0099 - val_loss: 200.4604 - val_mae: 10.3278
Epoch 2/50
63/63 [==============================] - 72s 1s/step - loss: 198.6523 - mae: 10.5766 - val_loss: 201.1941 - val_mae: 10.1786
Epoch 3/50
63/63 [==============================] - 72s 1s/step - loss: 198.6786 - mae: 10.5294 - val_loss: 202.3757 - val_mae: 10.5437
Epoch 4/50
63/63 [==============================] - 72s 1s/step - loss: 201.1111 - mae: 10.5697 - val_loss: 201.9453 - val_mae: 10.4320
Epoch 5/50
63/63 [==============================] - 67s 1s/step - loss: 200.5063 - mae: 10.5870 - val_loss: 200.0212 - val_mae: 10.3937
Epoch 6/50
63/63 [==============================] - 64s 1s/step - loss: 199.9755 - mae: 10.6473 - val_loss: 201.6517 - val_mae: 10.2566
Epoch 7/50
63/63 [==============================] - 67s 1s/step - loss: 198.9796 - mae: 10.5443 - val_loss: 199.5945 - val_mae: 10.2346
Epoch 8/50
63/63 [==============================] - 62s 979

In [16]:
from tensorflow.keras.preprocessing import image
import tensorflow as tf
import numpy as np
import os
import joblib

# Paths
MODEL_SAVE_PATH = r"C:\Users\priya\Downloads\Smart_Nutrition\models\food_model.h5"
SCALERS_PATH = r"C:\Users\priya\Downloads\Smart_Nutrition\models\nutrient_scalers.pkl"

# ===================== PREDICTION FUNCTIONS =====================
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict_nutrients(img_path):
    img_array = preprocess_image(img_path)
    pred_scaled = model.predict(img_array).flatten()
    pred_original = []
    for i, nutrient in enumerate(nutrient_columns):
        scaler = scalers[nutrient]
        value = scaler.inverse_transform([[pred_scaled[i]]])[0][0]
        pred_original.append(round(value,2))
    return dict(zip(nutrient_columns, pred_original))

thresholds = {"iron":5, "vitamin_b":0.5, "calcium":200, "vitamin_d":5,
              "vitamin_c":20, "protein":10, "fiber":5, "potassium":300}

def recommend_for_user(nutrient_dict, user_profile):
    recommendation = []
    points = 0
    total = 0
    for key, active in user_profile.items():
        if active:
            total += 1
            if nutrient_dict.get(key,0) >= thresholds[key]:
                points += 1
            else:
                recommendation.append(f"⚠️ Low in {key.title()}")
    score = round((points/total)*100,1) if total>0 else None
    if not recommendation:
        recommendation.append("✅ Meets your nutritional needs")
    return ", ".join(recommendation), score

# ===================== RUN EXAMPLE =====================
if __name__ == "__main__":
    if os.path.exists(MODEL_SAVE_PATH):
        print("✅ Loading trained model...")
        model = tf.keras.models.load_model(MODEL_SAVE_PATH)
        scalers = joblib.load(SCALERS_PATH)
        nutrient_columns = list(scalers.keys())  # Fix for prediction
    else:
        print("⚠️ No trained model found. Please run training first.")
        exit()

    # Test image
    test_img = r"C:\Users\priya\Downloads\Smart_Nutrition\nutrients\images\two-minute-breakfast-smoothie-4a4722d.jpg"
    nutrients = predict_nutrients(test_img)
    print("Predicted Nutrients:", nutrients)

    # Example user profile
    user_profile = {"protein": True, "iron": True, "vitamin_b": True}
    recommendation, score = recommend_for_user(nutrients, user_profile)
    print("Recommendation:", recommendation)
    print("Score:", score)


✅ Loading trained model...
1/1 [==============================] - 3s 3s/step
Predicted Nutrients: {'carbs': 36.7, 'fat': 14.52, 'fibre': 6.88, 'kcal': 355.3, 'protein': 17.19, 'salt': 0.72, 'saturates': 4.75, 'sugars': 13.22}
Recommendation: ⚠️ Low in Iron, ⚠️ Low in Vitamin_B
Score: 33.3
